# Phase 7 — Follow-up on two deferred items

Phases 3 and 4 each left something explicitly unfinished, on the record:

1. Phase 3: "No hyperparameter search backed these defaults" for gradient
   boosting.
2. Phase 4: `mean_bp` and `wbc` still violate the Cox proportional-hazards
   assumption after stratifying by `cancer`, "left for Phase 5 (binning, or a
   time-interaction term) rather than fixed here" — and Phase 5 never
   actually did it.

This notebook closes both out, honestly: one turns into a real, adopted
improvement; the other turns into a partial result that's disclosed rather
than force-fit. All model-fitting logic lives in `support_survival.models`;
this notebook only calls it and narrates.

In [1]:
from support_survival import data, features, evaluate, models, validate
from sklearn.metrics import roc_auc_score

df = data.load()
feat = features.build_features(df)
cols = evaluate.feature_columns(feat)
df.shape

(8873, 16)

## Part A — Hyperparameter search for gradient boosting

`models.tune_gradient_boosting` runs a randomized search (40 samples) over the
same knobs Phase 3 set by hand — tree count, depth, learning rate, both
subsampling rates — scored by AUROC under the identical 5-fold stratified CV
protocol Phase 3 used to compare models in the first place.

In [2]:
search = models.tune_gradient_boosting(feat[cols], feat["event"], cv=5, n_iter=40)
search.best_score_, search.best_params_

(np.float64(0.708098848911856),
 {'colsample_bytree': np.float64(0.836965827544817),
  'learning_rate': np.float64(0.01882557841679957),
  'max_depth': 4,
  'min_child_weight': 7,
  'n_estimators': 120,
  'subsample': np.float64(0.7801997007878172)})

**Reading this:** the search's best cross-validated AUROC beats Phase 3's
untuned baseline (0.698). That's encouraging, but a CV score from the same
search that produced it can be optimistic — the real test is Phase 5's
genuinely held-out split, never touched by this search.

In [3]:
train, val, test = validate.split_train_val_test(feat, test_size=0.2, val_size=0.2, random_state=42)

baseline = models.gradient_boosting()
tuned = models.gradient_boosting_tuned()  # the configuration `search` found above
baseline.fit(train[cols], train["event"])
tuned.fit(train[cols], train["event"])

proba_base = baseline.predict_proba(test[cols])[:, 1]
proba_tuned = tuned.predict_proba(test[cols])[:, 1]
y_test = test["event"].to_numpy()

{
    "baseline_auroc": roc_auc_score(y_test, proba_base),
    "tuned_auroc": roc_auc_score(y_test, proba_tuned),
    "baseline_brier": validate.brier(y_test, proba_base),
    "tuned_brier": validate.brier(y_test, proba_tuned),
}

{'baseline_auroc': 0.70049855887604,
 'tuned_auroc': 0.7110480821965763,
 'baseline_brier': 0.19388754665851593,
 'tuned_brier': 0.19238436222076416}

**Reading this:** on data neither model nor search ever saw, the tuned
configuration scores AUROC 0.711 vs 0.700 for the untuned baseline (matching
Phase 5's own number for the baseline exactly, since it's the same split) —
and Brier improves too, 0.192 vs 0.194. Both discrimination and calibration
moved the same direction, not a trade-off, which is the honest bar for
"tuning helped." Fewer, shallower trees with a lower learning rate
(`n_estimators=120` vs 300, `learning_rate=0.019` vs 0.05) generalize better
than Phase 3's reasoned-but-unsearched defaults.

**Decision:** `models.gradient_boosting_tuned()` replaces the untuned
configuration in `scripts/train_and_save.py` — this is what the served model
and the triage panel now use. Phase 3's own reported numbers are left
unchanged in the model card: they describe the untuned baseline that was
actually compared at the time, and stay reproducible from `gradient_boosting()`.

## Part B — The two disclosed Cox PH violations

Phase 4's primary model (dummy-coded cancer, stratified by nothing) showed
`mean_bp` and `wbc` still violating proportional hazards even after
stratifying by `cancer` resolved every other covariate. Reproducing that
check first:

In [4]:
df_cox = models.encode_cancer_stage(df)
vitals = ["age", "n_comorbidities", "mean_bp", "heart_rate", "serum_creatinine", "wbc"]
cph_baseline = models.fit_cox(df_cox, vitals, strata=["cancer"])
models.check_ph_assumption(cph_baseline, df_cox, vitals)

,test_statistic,p,-log2(p)
age,0.564125,0.452603,1.143682
heart_rate,2.177445,0.140047,2.836019
mean_bp,10.601172,0.001130,9.789257
n_comorbidities,0.063229,0.801464,0.319290
serum_creatinine,1.539249,0.214730,2.219407
wbc,15.682236,0.000075,13.704202


Confirmed: `mean_bp` (p ≈ 0.001) and `wbc` (p ≈ 0.00008) are the two real
violations. The standard remedy for a continuous covariate with a
non-proportional effect is to model it categorically instead — a step
function can absorb a shape that a single linear hazard multiplier can't.
Both get three clinically meaningful bands (implausible zeros, already
`NaN`-flagged, excluded rather than binned):

In [5]:
import numpy as np
import pandas as pd

banded = features.flag_implausible(df)
banded["mean_bp_band"] = pd.cut(banded["mean_bp"], bins=[-np.inf, 65, 110, np.inf], labels=["low", "normal", "high"])
banded["wbc_band"] = pd.cut(banded["wbc"], bins=[-np.inf, 4, 11, np.inf], labels=["low", "normal", "high"])
banded["mean_bp_low"] = (banded["mean_bp_band"] == "low").astype(float)
banded["mean_bp_high"] = (banded["mean_bp_band"] == "high").astype(float)
banded["wbc_low"] = (banded["wbc_band"] == "low").astype(float)
banded["wbc_high"] = (banded["wbc_band"] == "high").astype(float)
banded["cancer"] = df["cancer"]  # needed for strata

band_covariates = ["age", "n_comorbidities", "mean_bp_low", "mean_bp_high",
                    "heart_rate", "serum_creatinine", "wbc_low", "wbc_high"]
complete = banded.dropna(subset=["mean_bp_band", "wbc_band", "heart_rate"])
print(f"{len(df) - len(complete)} rows excluded (implausible mean_bp/heart_rate/wbc zeros)")

cph_banded = models.fit_cox(complete, band_covariates, strata=["cancer"])
models.check_ph_assumption(cph_banded, complete, band_covariates)

96 rows excluded (implausible mean_bp/heart_rate/wbc zeros)


,test_statistic,p,-log2(p)
age,1.277510,0.258363,1.952531
heart_rate,8.816985,0.002984,8.388346
mean_bp_high,1.740610,0.187062,2.418415
mean_bp_low,1.039559,0.307924,1.699354
n_comorbidities,0.103810,0.747304,0.420233
serum_creatinine,1.132802,0.287178,1.799981
wbc_high,4.070951,0.043627,4.518645
wbc_low,22.857889,0.000002,19.128905


**Reading this — a genuinely mixed result, reported as such:**
`mean_bp_low` (p ≈ 0.31) and `mean_bp_high` (p ≈ 0.19) both clear 0.05
comfortably — the categorical treatment resolves `mean_bp`'s violation.
`wbc_high` is borderline (p ≈ 0.044) and `wbc_low` is not resolved at all
(p ≈ 0.000002) — banding didn't fix `wbc`. A third covariate, `heart_rate`,
newly shows a violation here (p ≈ 0.003) that wasn't flagged in the original
continuous-covariate check; the 96 excluded rows (implausible `mean_bp`,
`heart_rate`, or `wbc` zeros) change the sample slightly, and this is
reported rather than quietly dropped. Concordance is essentially unchanged
(0.585 vs 0.583 for this vitals-only stratified comparison), so nothing was
lost switching `mean_bp` to bands, and nothing was gained trying the same
fix for `wbc`.

Plausible reason `wbc` resists this fix where `mean_bp` doesn't: a single
low blood pressure reading is likely closer to noise around a fairly stable
physiological state, while an abnormal white blood cell count (especially
low, e.g. from chemotherapy-induced neutropenia) can mark a *trajectory* —
its relationship to risk plausibly does change shape over the follow-up
period in a way three static bands still can't capture. Fully resolving that
would need a genuine time-varying-covariate model
(lifelines' `CoxTimeVaryingFitter`, restructuring each patient's follow-up
into time-sliced episodes) — a materially bigger undertaking than this
follow-up, and out of scope here.

**Decision:** do not change Phase 4's primary served model. Swapping in
bands fixes one violation, surfaces another, and doesn't fix the one it was
meant to address, for no concordance gain — there's no clean win to adopt,
and touching an already-validated model for a partial, unproven improvement
is exactly the kind of undisciplined change this project has tried to avoid
elsewhere. This investigation, and its partial result, is disclosed here and
in the model card instead.

## Summary

- **Adopted:** a tuned gradient boosting configuration (found via a 40-sample
  randomized search under Phase 3's own CV protocol), verified on Phase 5's
  untouched held-out test set: AUROC 0.700 → 0.711, Brier 0.194 → 0.192.
  Now the model `scripts/train_and_save.py` ships.
- **Investigated, not adopted:** categorical binning for `mean_bp`/`wbc` to
  address Phase 4's disclosed PH violations. Fixes `mean_bp`, doesn't fix
  `wbc`, no concordance gain — so Phase 4's primary model is unchanged, and
  a full fix (time-varying covariates) is noted as a real but materially
  larger undertaking than this follow-up.